# FGSM — Explaining and Harnessing Adversarial Examples (2015)

_Papers · Meta-learning, Bayesian & Robustness_

**A tiny perturbation in the gradient-sign direction flips a classifier; training on those examples is the fix.**

---

This notebook is a practice scaffold for the **FGSM — Explaining and Harnessing Adversarial Examples (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the worked example: FGSM on a 3-pixel linear classifier. ---
# Score s = w.x + b for the positive class; loss J = softplus(-s) (label y=1).
# grad_x J = -sigmoid(-s)*w  ->  eta = eps*sign(grad_x J).
w = torch.tensor([2.0, -3.0, 0.5]); b = torch.tensor(1.0)
x = torch.tensor([1.0, 1.0, 1.0], requires_grad=True)
s = (w*x).sum() + b
J = F.softplus(-s)                                   # -log sigmoid(s), target class 1
g, = torch.autograd.grad(J, x)
eta = 0.1 * torch.sign(g)
print("worked example: s=%.3f  grad_x J=%s" % (s.item(), [round(v,4) for v in g.tolist()]))
print("  sign=%s  eta(eps=0.1)=%s" % ([int(v) for v in torch.sign(g).tolist()],
                                      [round(v,2) for v in eta.tolist()]))
s_adv = (w*(x.detach()+eta)).sum() + b
print("  score before=%.3f  after FGSM=%.3f  (w.eta = -eps*sum|w| = %.2f)" % (
      s.item(), s_adv.item(), -0.1*w.abs().sum().item()))
# worked example: s=0.500  grad_x J=[-0.755, 1.1326, -0.1888]
#   sign=[-1, 1, -1]  eta(eps=0.1)=[-0.1, 0.1, -0.1]
#   score before=0.500  after FGSM=-0.050  (w.eta = -eps*sum|w| = -0.55)


# --- 1. Data: 8x8 sklearn digits (no download), pixels scaled to [0,1]. ---
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True); X = X.astype(np.float32)/16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xtr, ytr = torch.tensor(Xtr), torch.tensor(ytr)
Xte, yte = torch.tensor(Xte), torch.tensor(yte)

# --- 2. The model: a tiny multi-layer perceptron, composed with torch.nn. ---
def make_net():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,10))

# --- 3. FGSM, built by hand: gradient w.r.t. the INPUT, then eps*sign (Sec 4). ---
def fgsm(net, x, y, eps):
    x = x.clone().detach().requires_grad_(True)
    loss = F.cross_entropy(net(x), y)
    g, = torch.autograd.grad(loss, x)                # grad of loss w.r.t. INPUT x
    return (x + eps*g.sign()).clamp(0, 1).detach()   # eps*sign, clamp to valid range

def acc(net, x, y):
    with torch.no_grad():
        return (net(x).argmax(1) == y).float().mean().item()

# --- 4. Train: clean, and adversarially with mixed loss (Sec 6, alpha=0.5). ---
def train(adversarial, eps=0.25, epochs=120):
    net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(net(Xtr), ytr)        # J_clean
        if adversarial:                              # J~ = 0.5*J_clean + 0.5*J_adv
            Xadv = fgsm(net, Xtr, ytr, eps)          # regenerated every step
            loss = 0.5*loss + 0.5*F.cross_entropy(net(Xadv), ytr)
        loss.backward(); opt.step()
    return net

clean = train(adversarial=False)
advt  = train(adversarial=True, eps=0.25)

# --- 5. Sweep epsilon: FGSM accuracy for clean vs adversarially-trained models. ---
print("\neps    clean-trained acc   adv-trained acc")
for e in [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    if e == 0.0:
        ca, aa = acc(clean, Xte, yte), acc(advt, Xte, yte)
    else:
        ca = acc(clean, fgsm(clean, Xte, yte, e), yte)
        aa = acc(advt,  fgsm(advt,  Xte, yte, e), yte)
    print("%.2f       %.4f            %.4f" % (e, ca, aa))
# eps    clean-trained acc   adv-trained acc
# 0.00       0.9407            0.9407      <- identical clean accuracy
# 0.10       0.5815            0.7352
# 0.25       0.0000            0.1870      <- attack wrecks clean model; defense holds some
# 0.30       0.0000            0.0463
# (Our small run on 8x8 digits, NOT the paper's MNIST numbers.)

## Visualize the data & results

_As we raise the FGSM perturbation budget epsilon, how fast does test accuracy fall for a normally-trained model versus one hardened with adversarial training?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True); X = X.astype(np.float32)/16.0   # 8x8, pixels in [0,1]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xtr, ytr = torch.tensor(Xtr), torch.tensor(ytr)
Xte, yte = torch.tensor(Xte), torch.tensor(yte)

def make_net():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,10))

def fgsm(net, x, y, eps):                              # FGSM, Sec 4
    x = x.clone().detach().requires_grad_(True)
    g, = torch.autograd.grad(F.cross_entropy(net(x), y), x)   # grad w.r.t. INPUT
    return (x + eps*g.sign()).clamp(0,1).detach()

def acc(net, x, y):
    with torch.no_grad(): return (net(x).argmax(1)==y).float().mean().item()

def train(adversarial, eps=0.25, epochs=120):         # Sec 6 mixed loss, alpha=0.5
    net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad(); loss = F.cross_entropy(net(Xtr), ytr)
        if adversarial:
            Xadv = fgsm(net, Xtr, ytr, eps)
            loss = 0.5*loss + 0.5*F.cross_entropy(net(Xadv), ytr)
        loss.backward(); opt.step()
    return net

clean = train(adversarial=False); advt = train(adversarial=True, eps=0.25)
eps_grid = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
clean_c = [acc(clean, Xte, yte) if e==0 else acc(clean, fgsm(clean,Xte,yte,e), yte) for e in eps_grid]
adv_c   = [acc(advt,  Xte, yte) if e==0 else acc(advt,  fgsm(advt, Xte,yte,e), yte) for e in eps_grid]
print("eps   :", eps_grid)
print("clean :", [round(v,4) for v in clean_c])
print("adv   :", [round(v,4) for v in adv_c])
# eps   : [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
# clean : [0.9407, 0.8278, 0.5815, 0.2667, 0.0556, 0.0, 0.0]
# adv   : [0.9407, 0.8537, 0.7352, 0.5611, 0.3722, 0.187, 0.0463]
# At eps=0.25 the clean model is at 0% while the adv-trained model holds 18.7%.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The defense ablation. You have working adversarial training with $\alpha=0.5$. Set
            $\alpha=1$ (use only the clean loss, drop the FGSM term) and retrain, everything else identical.
            What did you just remove, and what do you expect to happen to accuracy on FGSM examples at
            $\epsilon=0.25$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the mixed loss loss = 0.5*J_clean + 0.5*J_adv and change it to
                 loss = J_clean (equivalently $\alpha=1$). — _$\alpha=1$ zeroes the
                 $(1-\alpha)J_{\text{adv}}$ term, so the network never sees its own FGSM examples during
                 training &mdash; it reduces to ordinary clean training._
- Identify what was dropped: the worst-case neighbor term that forces correctness on a shell
                 around each input. — _Without it, the network optimizes only the data points, leaving the
                 same linear directions exploitable by FGSM._
- Retrain and re-attack at $\epsilon=0.25$: expect accuracy on adversarial examples to fall back
                 toward the clean-trained model's level. — _The robustness came specifically from training
                 on FGSM examples; remove that term and you remove the defense._

**Answer:** Setting $\alpha=1$ deletes the adversarial term $(1-\alpha)\,J_{\text{adv}}$, so training
                 reverts to clean-only &mdash; the network never practices on its own FGSM examples. Re-attacked
                 at $\epsilon=0.25$, its adversarial accuracy collapses back to roughly the clean-trained
                 model's level (near zero in our run). This confirms the recovery in the CODEVIZ panel came from
                 the FGSM term, not from anything else. The defense is the augmented loss.

</details>

**Problem 2.** Why does FGSM use $\text{sign}(\nabla_x J)$ rather than the gradient $\nabla_x J$ itself, or a
            normalized gradient $\nabla_x J / \lVert\nabla_x J\rVert_2$? Tie your answer to which norm the
            perturbation budget is measured in.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the linearized loss increase: $(\nabla_x J)^\top\eta$, to be maximized subject to a
                 budget on $\eta$. — _Locally the loss change is this dot product; FGSM maximizes it under
                 a size constraint._
- State the budget: $\lVert\eta\rVert_\infty \le \epsilon$ &mdash; no single pixel moves more
                 than $\epsilon$. — _This is a max-norm (per-pixel) cap, the natural "imperceptible to a
                 human" constraint for images._
- Maximize term by term: each $\eta_j$ should match its gradient's sign and take the full
                 $\epsilon$. The maximizer is $\epsilon\,\text{sign}(\nabla_x J)$. — _Under a per-pixel
                 cap the magnitude of the gradient is irrelevant &mdash; only its sign and the cap $\epsilon$
                 matter._

**Answer:** Because the budget is the max-norm $\lVert\eta\rVert_\infty \le \epsilon$: every
                 pixel may move at most $\epsilon$. To make the loss climb as much as possible under that
                 per-pixel cap, push each pixel the full $\epsilon$ in the direction its gradient points
                 &mdash; that is exactly $\epsilon\,\text{sign}(\nabla_x J)$. The gradient's magnitude
                 does not matter, so the raw or $L_2$-normalized gradient would be the wrong shape for this
                 budget (those maximize the loss under an $L_2$ budget instead). The sign is the closed-form
                 max-norm maximizer.

</details>

**Problem 3.** In the worked example you had $w=[2,-3,0.5]$, $b=1$, $x=[1,1,1]$, $\epsilon=0.1$, giving score
            $s=0.5$ and adversarial score $s_{\text{adv}}=-0.05$. Suppose instead the input had $n=12$ pixels,
            each weight $\pm$ a similar magnitude with $\sum_j|w_j| = 22$. With the same $\epsilon=0.1$, how
            much does the worst-case perturbation move the score, and what does this say about dimension?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the worst-case score shift: $w^\top\eta$ with $\eta=\epsilon\,\text{sign}(w)$
                 pointed to decrease the positive-class score gives $-\epsilon\sum_j|w_j|$. — _Each
                 pixel contributes $\epsilon|w_j|$ to the shift; summing gives $\epsilon\sum|w_j|$._
- Plug in: $-0.1 \times 22 = -2.2$. — _The score can be driven from its starting value down
                 by $2.2$ &mdash; far past any reasonable decision boundary._
- Compare to the 3-pixel case ($\sum|w_j|=5.5$, shift $-0.55$). Same per-pixel budget
                 $\epsilon=0.1$, but the shift grew with $\sum|w_j|=mn$. — _This is the &sect;3 argument:
                 output change $\epsilon m n$ scales with dimension $n$, the perturbation size $\epsilon$ does
                 not._

**Answer:** The worst-case shift is $-\epsilon\sum_j|w_j| = -0.1\times 22 = -2.2$ &mdash; four times the
                 $-0.55$ shift in the 3-pixel case, from the same per-pixel budget $\epsilon=0.1$. This
                 is exactly the dimensionality argument of &sect;3: $w^\top\eta = \epsilon m n$ grows with the
                 input dimension $n$ (via $\sum|w_j| = mn$), while the perturbation's max-norm stays fixed at
                 $\epsilon$. High-dimensional inputs are easier to attack with imperceptible perturbations
                 precisely because the tiny per-pixel nudges accumulate.

</details>